# 21 — MVCC & Isolation

### Why this exists
MVCC lets readers observe a stable version while writers create newer versions. This lab simulates snapshots and exposes how isolation choices change what a transaction can see.

### Learning objectives
represent versions with commit timestamps; read snapshots; simulate concurrent writers

### Prerequisite
Python tuples and sorting

### Engineering loop
**Predict → Build → Observe → Break → Diagnose → Improve → Generalize → Defend**

## 🖊️ Sketch note
![Sketch note](../assets/sketch-notes/21-mvcc-isolation.svg)

> **Question to carry through the lab:** If transaction T1 starts before T2 commits, should T1 see T2's write? What changes under a newer snapshot?

## Capability contract

**Capability:** C05 — Storage Consistency & Distributed Systems  
**Workstream:** Distributed Data Platform  
**Primary roles:** Data Engineer|Platform Engineer|Architect  
**You will prove:** Explain a distributed trade-off


## Mental model
MVCC separates logical rows from versions. A transaction reads according to a visibility rule over committed versions; isolation determines which changes may become visible during its lifetime.

## 1. Predict before you run
Write down what you expect and what evidence would prove you wrong. Do not change the experiment after seeing the result.

## 2. Build
The first cell creates the smallest system that can answer the question. The later cells change one boundary at a time.

## Agent-native lens

**Agent can do:** routine implementation or a first-pass analysis.

**You must understand:** the mechanism and its assumptions.

**You must verify:** use tests, measurements or adversarial data that could prove the output wrong.

**You must decide:** the trade-off, failure response or production boundary.


## Apprenticeship bridge

Use this notebook to prepare for the following Acme Commerce work scenario(s). The scenario gives you symptoms and constraints; it does **not** give you the diagnosis.

- **S08 — Replication and consistency (DATA-1110)** → [`../work_simulations/08_acme_commerce_replica_lag/README.md`](../work_simulations/08_acme_commerce_replica_lag/README.md)

**How to use the bridge:** finish the experiment and independent challenge here, then enter the ticket without reopening the notebook as an answer key. Bring your own prediction, evidence and verification plan.


In [ ]:
versions=[
 (1,"t0","price",100,True),
 (2,"t1","price",110,True),
 (3,"t2","price",120,False),
]
def visible(key,snapshot):
    candidates=[v for v in versions if v[2]==key and v[0]<=snapshot and v[4]]
    return max(candidates)[3] if candidates else None
for s in [1,2,3]: print(s,visible("price",s))

In [ ]:
# T1 starts at snapshot 2; T2 commits version 3 later.
t1_snapshot=2
print("T1 first read",visible("price",t1_snapshot))
versions[-1]=(3,"t2","price",120,True)
print("T1 repeatable snapshot",visible("price",t1_snapshot))
print("new transaction",visible("price",3))

In [ ]:
# Break: delete the older version and observe what a long-running snapshot loses.
truncated=[v for v in versions if not (v[0]==2 and v[2]=="price")]
print("versions retained",truncated)
print("snapshot 2 after premature cleanup",max([v for v in truncated if v[2]=="price" and v[0]<=2], default=None))

In [ ]:
# Improve: garbage collection must respect the oldest active snapshot.
oldest_active=2
retained=[v for v in versions if v[0]>=oldest_active or v[0]==1]
print("safe retention",retained)

## 3. Measure
Do not stop at “it worked”. Record a correctness signal, a failure signal, and at least one quantitative measurement where the concept permits it.

## 4. Break deliberately
Introduce one controlled failure. Keep the failure reproducible so another engineer can reproduce the observation.

## 5. Diagnose
Use evidence, not the implementation you expected. State: **symptom → mechanism → evidence → boundary**. If two explanations fit, design one more measurement.

## 6. Improve
Make the smallest change that addresses the mechanism. Re-run the original experiment and the failure case.

## 7. Generalize
Ask what changes at 10× load, with retries, multiple writers/readers, partial failure, or changing contracts. Separate what this toy proves from what it only illustrates.

## 8. Production bridge
Production MVCC involves transaction IDs, vacuum/GC, lock interaction, snapshots, serialization rules and workload-specific anomaly analysis.

## 9. Independent challenge
Add a second key and two transactions to illustrate a write skew or lost-update scenario. State which isolation property prevents it and which property the toy model cannot prove.

### Evidence to keep
snapshot reads, visibility before/after commit, unsafe cleanup observation, and an explicit isolation decision

## 10. Explain it in 60–90 seconds
Explain the mechanism, your measurement, the failure you injected, the diagnosis, and the production decision you would make. Avoid tool names until the mental model is clear.